# Federated Unlearning for Vision Models with Auditing & Visualization
### Implementing the **Right to be Forgotten** via **FedEraser** on CIFAR-100

---
## 🎯 Objective & Problem Formulation
Under privacy regulations such as **GDPR Article 17** ("Right to be Forgotten"), clients participating in Federated Learning (FL) have the legal right to revoke their data and have all trace of their contributions removed from the global model.

In this notebook, we demonstrate an end-to-end Federated Unlearning pipeline:
1. **Single-Class-Per-Client Setup**: 10 clients on CIFAR-100, where Client $ holds exclusively Class $ samples.
2. **Flower () Integration**: Federated client-server communication using .
3. **Original FL Model ({	ext{orig}}$)**: Trained across all 10 clients with round-by-round update logging.
4. **Retrained Gold Standard ({	ext{retrain}}$)**: Retrained from scratch excluding the forgotten client.
5. **FedEraser Unlearning ({	ext{unlearn}}$)**: Fast unlearning via calibrated trajectory reconstruction.
6. **Grad-CAM Visual Evidence**: Attribution heatmaps demonstrating visual erasure of the forgotten concept.


In [ ]:
import os, sys
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.data.cifar_loader import SingleClassFLDataManager
from src.models.resnet_cifar import get_cifar_model
from src.flwr_fl.runner import run_fl_training, eval_client_split
from src.unlearning.federaser import FedEraserEngine
from src.visualize.gradcam_visualizer import GradCAMExplainer, generate_unlearning_gallery

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}) ")

## 1. CIFAR-100 Data Partitioning (1 Client = 1 Class)
We partition CIFAR-100 such that Client 0 holds Class 0 (), Client 1 holds Class 1 (), ..., and Client 9 holds Class 9 ().
Client 9 will simulate the **Right to be Forgotten** request.

In [ ]:
num_classes = 10
forget_client = 9
data_manager = SingleClassFLDataManager(
    root="../dataset",
    selected_classes=list(range(num_classes)),
    batch_size=32,
    num_workers=0
)

print(f"Target Forgotten Client: Client {forget_client} ('{data_manager.class_names[forget_client]}')")
print(f"Retained Clients: {[c for c in range(num_classes) if c != forget_client]}")

## 2. Phase 1: Federated Training of Original Baseline Model ({	ext{orig}}$)
We train the original FL model across all 10 clients using Flower ().
The central server records historical global checkpoints , \dots, W_T$ and client updates $\Delta W_{t, k}$ for FedEraser.

In [ ]:
if os.path.exists("../checkpoints/M_orig.pt") and os.path.exists("../checkpoints/fl_training_cache.pt"):
    print("Loading cached original FL model and training history...")
    cache = torch.load("../checkpoints/fl_training_cache.pt", map_location=device)
    hist_globals = cache["history_global_models"]
    hist_updates = cache["history_client_updates"]
    orig_history = cache["history"]
    orig_model = get_cifar_model(num_classes=num_classes, device=device)
    orig_model.load_state_dict(torch.load("../checkpoints/M_orig.pt", map_location=device))
    print(f"Loaded M_orig successfully ({len(hist_globals)} checkpoint rounds logged).")
else:
    orig_model, orig_history, hist_globals, hist_updates = run_fl_training(
        data_manager=data_manager,
        client_ids=list(range(num_classes)),
        num_rounds=6,
        local_epochs=2,
        learning_rate=0.001,
        device=device,
        save_checkpoint_path="../checkpoints/M_orig.pt",
        forget_client_id=forget_client,
        tag="Original FL"
    )


## 3. Phase 2: Exact Retraining Gold Standard Baseline ({	ext{retrain}}$)
Exact retraining completely removes Client 9 and retrains across all rounds using only the 9 retained clients.

In [ ]:
engine = FedEraserEngine(data_manager=data_manager, forget_client_id=forget_client, device=device)
if os.path.exists("../checkpoints/M_retrain.pt"):
    print("Loading pre-computed Retrained Gold Standard model...")
    retrain_model = get_cifar_model(num_classes=num_classes, device=device)
    retrain_model.load_state_dict(torch.load("../checkpoints/M_retrain.pt", map_location=device))
    # Evaluation metrics from benchmark
    with open("../results/unlearning_benchmark.json") as bf:
        bench = json.load(bf)
    retrain_metrics = bench["models"]["retrained"]
    print(f"Loaded M_retrain (Benchmark Retain Acc: {retrain_metrics['retain_acc']:.2%}, Runtime: {retrain_metrics['runtime_seconds']:.1f}s)")
else:
    retrain_model, retrain_metrics, retrain_history = engine.run_exact_retrain(
        num_rounds=6,
        local_epochs=2,
        learning_rate=0.001,
        save_path="../checkpoints/M_retrain.pt"
    )


## 4. Phase 3: Fast Federated Unlearning via FedEraser ({	ext{unlearn}}$)
FedEraser reconstructs the unlearned model along the historical training path using fast calibration steps on retained clients.

In [ ]:
if os.path.exists("../checkpoints/M_unlearn.pt"):
    print("Loading pre-computed FedEraser unlearned model...")
    unlearn_model = get_cifar_model(num_classes=num_classes, device=device)
    unlearn_model.load_state_dict(torch.load("../checkpoints/M_unlearn.pt", map_location=device))
    with open("../results/unlearning_benchmark.json") as bf:
        bench = json.load(bf)
    unlearn_metrics = bench["models"]["federaser"]
    print(f"Loaded M_unlearn (Benchmark Retain Acc: {unlearn_metrics['retain_acc']:.2%}, Runtime: {unlearn_metrics['runtime_seconds']:.1f}s, Speedup: {unlearn_metrics['speedup_vs_retrain']:.2f}x)")
else:
    unlearn_model, unlearn_metrics, unlearn_history = engine.run_federaser(
        history_global_models=hist_globals,
        history_client_updates=hist_updates,
        cali_epochs=1,
        cali_steps=4,
        cali_lr=0.001,
        milestone_interval=2,
        save_path="../checkpoints/M_unlearn.pt"
    )


## 5. Phase 4: Final Benchmark Comparison Table
We evaluate all three models on holdout CIFAR-100 test data across Retain and Forget splits.

In [ ]:
full_test_loader = data_manager.get_full_test_loader()
eval_orig = eval_client_split(orig_model, full_test_loader, forget_client, device=device)
eval_ret = eval_client_split(retrain_model, full_test_loader, forget_client, device=device)
eval_unl = eval_client_split(unlearn_model, full_test_loader, forget_client, device=device)

speedup = retrain_metrics["runtime_seconds"] / max(0.01, unlearn_metrics["runtime_seconds"])

print(f"{'Model':<25} | {'Retain Acc':<10} | {'Forget Acc':<10} | {'Forget Conf':<11} | {'Speedup':<10}")
print("-" * 75)
print(f"{'Original FL (M_orig)':<25} | {eval_orig['retain_acc']:>9.2%} | {eval_orig['forget_acc']:>9.2%} | {eval_orig['forget_conf']:>10.3f} | 1.0x")
print(f"{'FedEraser (M_unlearn)':<25} | {eval_unl['retain_acc']:>9.2%} | {eval_unl['forget_acc']:>9.2%} | {eval_unl['forget_conf']:>10.3f} | {speedup:.2f}x")
print(f"{'Retrained Gold (M_retrain)':<25} | {eval_ret['retain_acc']:>9.2%} | {eval_ret['forget_acc']:>9.2%} | {eval_ret['forget_conf']:>10.3f} | (Baseline)")

## 6. Phase 5: Visual Attribution Evidence (Grad-CAM)
We extract Grad-CAM heatmaps from the convolutional target layer () to visually prove what the model has forgotten.

In [ ]:
forget_imgs, forget_lbls = data_manager.get_sample_images_for_class(forget_client, count=2)
retain_imgs, retain_lbls = data_manager.get_sample_images_for_class(0, count=2)
samples = torch.cat([forget_imgs, retain_imgs], dim=0)
labels = forget_lbls + retain_lbls

gallery_res = generate_unlearning_gallery(
    models_dict={"Original": orig_model, "FedEraser": unlearn_model, "Retrained": retrain_model},
    sample_images=samples,
    sample_labels=labels,
    class_names=data_manager.class_names,
    unnormalizer=data_manager.unnormalizer,
    forget_client_id=forget_client,
    save_path="../results/gradcam_notebook_gallery.png",
    device=device
)

import matplotlib.image as mpimg
plt.figure(figsize=(15, 12))
plt.imshow(mpimg.imread("../results/gradcam_notebook_gallery.png"))
plt.axis("off")
plt.title("Grad-CAM Attention Comparison Gallery", fontsize=14, fontweight="bold")
plt.show()